# Predicting Warping Defects in FDM 3D-Printed Components — Colab Notebook

**Goal:** Predict whether a printing job is likely to develop a warping defect before printing.

This notebook performs dataset upload, version comparison, preprocessing, feature engineering, feature-store creation, Logistic Regression and Random Forest training, evaluation, MLflow tracking/registration, and a FastAPI prediction demonstration.

**Important:** These are synthetic educational data. The risk labels are generated from plausible material-dependent rules, not real printer measurements. Treat the model as a course demonstration, not production-grade engineering advice.

## 1. Install required libraries

In [1]:
!pip -q install mlflow fastapi uvicorn scikit-learn pandas numpy joblib

## 2. Upload the datasets

In [2]:
from google.colab import files
uploaded = files.upload()  # Upload printing_jobs_v1.csv and printing_jobs_v2.csv

import pandas as pd
from pathlib import Path

required_files = ["printing_jobs_v1.csv", "printing_jobs_v2.csv"]
missing_files = [name for name in required_files if name not in uploaded]
if missing_files:
    raise FileNotFoundError(
        f"Please upload both files: {missing_files}. "
        "Download them from the provided dataset links and run this cell again."
    )

v1 = pd.read_csv("printing_jobs_v1.csv")
v2 = pd.read_csv("printing_jobs_v2.csv")
print("Version 1 shape:", v1.shape)
print("Version 2 shape:", v2.shape)
display(v1.head())

Saving printing_jobs_v1.csv to printing_jobs_v1.csv
Saving printing_jobs_v2.csv to printing_jobs_v2.csv
Version 1 shape: (400, 10)
Version 2 shape: (400, 10)


,job_id,filament_material,nozzle_temperature_c,bed_temperature_c,printing_speed_mm_s,ambient_temperature_c,cooling_fan_speed_pct,component_base_area_cm2,layer_height_mm,warping_defect
0,V1_0001,ABS,253.4,112.6,51.5,20.8,43.0,54.7,0.30,No
1,V1_0002,PLA,210.0,59.1,57.7,18.9,51.5,138.4,0.25,Yes
2,V1_0003,PLA,201.9,57.3,54.7,18.7,64.6,92.3,0.20,Yes
3,V1_0004,PETG,232.3,74.7,51.3,24.6,69.3,80.0,0.10,No
4,V1_0005,ABS,236.5,102.1,47.0,21.6,40.5,161.8,0.10,Yes


## 3. Validate datasets and compare versions

In [3]:
required_columns = [
    "job_id", "filament_material", "nozzle_temperature_c",
    "bed_temperature_c", "printing_speed_mm_s", "ambient_temperature_c",
    "cooling_fan_speed_pct", "component_base_area_cm2",
    "layer_height_mm", "warping_defect"
]
for version_name, frame in [("V1", v1), ("V2", v2)]:
    missing = sorted(set(required_columns) - set(frame.columns))
    assert not missing, f"{version_name} missing columns: {missing}"
    assert set(frame["filament_material"].dropna().unique()).issubset({"PLA","ABS","PETG"})
    assert set(frame["warping_defect"].dropna().unique()).issubset({"Yes","No"})

comparison = pd.DataFrame({
    "Version": ["V1", "V2"],
    "Records": [len(v1), len(v2)],
    "Mean printing speed (mm/s)": [v1.printing_speed_mm_s.mean(), v2.printing_speed_mm_s.mean()],
    "Mean bed temperature (°C)": [v1.bed_temperature_c.mean(), v2.bed_temperature_c.mean()],
    "Warping rate (%)": [(v1.warping_defect == "Yes").mean()*100,
                         (v2.warping_defect == "Yes").mean()*100]
}).round(2)
display(comparison)

,Version,Records,Mean printing speed (mm/s),Mean bed temperature (°C),Warping rate (%)
0,V1,400,55.45,76.92,33.25
1,V2,400,79.71,66.87,62.50


## 4. Prepare Version 1 and engineer features

In [4]:
import numpy as np

MATERIAL_STRESS = {"PLA": 0.75, "ABS": 1.25, "PETG": 1.0}
RAW_FEATURES = [
    "filament_material", "nozzle_temperature_c", "bed_temperature_c",
    "printing_speed_mm_s", "ambient_temperature_c", "cooling_fan_speed_pct",
    "component_base_area_cm2", "layer_height_mm"
]
NUMERIC_FEATURES = [
    "nozzle_temperature_c", "bed_temperature_c", "printing_speed_mm_s",
    "ambient_temperature_c", "cooling_fan_speed_pct",
    "component_base_area_cm2", "layer_height_mm",
    "nozzle_bed_temperature_difference", "cooling_stress_index"
]
CATEGORICAL_FEATURES = ["filament_material"]
TARGET = "warping_defect"

def engineer_features(frame):
    df = frame.copy()
    df["nozzle_bed_temperature_difference"] = (
        df["nozzle_temperature_c"] - df["bed_temperature_c"]
    )
    material_factor = df["filament_material"].map(MATERIAL_STRESS).fillna(1.0)
    thermal_gap = df["nozzle_bed_temperature_difference"].fillna(0).clip(lower=0)
    fan_factor = 1 + df["cooling_fan_speed_pct"].fillna(0)/100
    speed_factor = 1 + df["printing_speed_mm_s"].fillna(0)/150
    df["cooling_stress_index"] = thermal_gap * fan_factor * speed_factor * material_factor
    return df

engineered_v1 = engineer_features(v1)
feature_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Save engineered features separately as a simple reusable feature store.
feature_store = engineered_v1[["job_id"] + feature_columns + [TARGET]].copy()
feature_store.to_csv("feature_store_v1.csv", index=False)
print("Saved feature_store_v1.csv")
display(feature_store.head())

Saved feature_store_v1.csv


,job_id,nozzle_temperature_c,bed_temperature_c,printing_speed_mm_s,ambient_temperature_c,cooling_fan_speed_pct,component_base_area_cm2,layer_height_mm,nozzle_bed_temperature_difference,cooling_stress_index,filament_material,warping_defect
0,V1_0001,253.4,112.6,51.5,20.8,43.0,54.7,0.30,140.8,338.090133,ABS,No
1,V1_0002,210.0,59.1,57.7,18.9,51.5,138.4,0.25,150.9,237.415120,PLA,Yes
2,V1_0003,201.9,57.3,54.7,18.7,64.6,92.3,0.20,144.6,243.604873,PLA,Yes
3,V1_0004,232.3,74.7,51.3,24.6,69.3,80.0,0.10,157.6,358.068146,PETG,No
4,V1_0005,236.5,102.1,47.0,21.6,40.5,161.8,0.10,134.4,309.999200,ABS,Yes


## 5. Build preprocessing and train/test split

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

X = engineered_v1[feature_columns]
y = engineered_v1[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES)
])
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

Training rows: 300 | Test rows: 100


## 6. Train Logistic Regression and Random Forest

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.base import clone

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, max_depth=10, min_samples_leaf=2,
        class_weight="balanced", random_state=42
    )
}
trained_models = {}
rows = []

for name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    metrics = {
        "Model": name,
        "Precision": precision_score(y_test, predictions, pos_label="Yes", zero_division=0),
        "Recall": recall_score(y_test, predictions, pos_label="Yes", zero_division=0),
        "F1-score": f1_score(y_test, predictions, pos_label="Yes", zero_division=0)
    }
    rows.append(metrics)
    trained_models[name] = pipeline
    print("\n", name)
    print(classification_report(y_test, predictions, zero_division=0))
    print("Confusion matrix (labels No, Yes):\n",
          confusion_matrix(y_test, predictions, labels=["No","Yes"]))

results = pd.DataFrame(rows).sort_values("F1-score", ascending=False)
display(results)
best_name = results.iloc[0]["Model"]
best_model = trained_models[best_name]
print("Selected model:", best_name)


 Logistic Regression
              precision    recall  f1-score   support

          No       0.72      0.51      0.60        67
         Yes       0.38      0.61      0.47        33

    accuracy                           0.54       100
   macro avg       0.55      0.56      0.53       100
weighted avg       0.61      0.54      0.55       100

Confusion matrix (labels No, Yes):
 [[34 33]
 [13 20]]

 Random Forest
              precision    recall  f1-score   support

          No       0.67      0.78      0.72        67
         Yes       0.32      0.21      0.25        33

    accuracy                           0.59       100
   macro avg       0.49      0.49      0.49       100
weighted avg       0.55      0.59      0.56       100

Confusion matrix (labels No, Yes):
 [[52 15]
 [26  7]]


,Model,Precision,Recall,F1-score
0,Logistic Regression,0.377358,0.606061,0.465116
1,Random Forest,0.318182,0.212121,0.254545


Selected model: Logistic Regression


## 7. Save model and register it with MLflow

In [8]:

import os
import joblib
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Create artifacts folder
os.makedirs("artifacts", exist_ok=True)

# Save the selected model locally
joblib.dump(best_model, "artifacts/best_model.joblib")

# Save model comparison results
results.to_csv("model_comparison.csv", index=False)

# Configure MLflow tracking
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("FDM_Warping_Defect_Prediction")

# Prepare model signature
signature = infer_signature(
    X_test,
    best_model.predict(X_test)
)

# Log the selected model
with mlflow.start_run(
    run_name=best_name.replace(" ", "_")
) as run:

    mlflow.log_param("selected_model", best_name)
    mlflow.log_param("dataset_version", "v1")

    selected_row = results.iloc[0]

    mlflow.log_metric(
        "precision", float(selected_row["Precision"])
    )
    mlflow.log_metric(
        "recall", float(selected_row["Recall"])
    )
    mlflow.log_metric(
        "f1_score", float(selected_row["F1-score"])
    )

    # Use pickle instead of skops serialization
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="model",
        serialization_format="pickle",
        signature=signature,
        input_example=X_test.head(3)
    )

    model_uri = f"runs:/{run.info.run_id}/model"

# Register the model after logging
registered = mlflow.register_model(
    model_uri,
    "FDM_Warping_Predictor"
)

print("MLflow Run ID:", run.info.run_id)
print("Registered Model Version:", registered.version)
print("Selected Model:", best_name)
print("Best model saved successfully!")


2026/11/18 08:50:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'FDM_Warping_Predictor'.
2026/11/18 08:50:56 WARNING mlflow.tracking._model_registry.fluent: Run with id 27e365abab6841e89fa15d685a2abe52 has no artifacts at artifact path 'model', registering model based on models:/m-586e420157a242df99073cadf4ba5488 instead


MLflow Run ID: 27e365abab6841e89fa15d685a2abe52
Registered Model Version: 1
Selected Model: Logistic Regression
Best model saved successfully!


Created version '1' of model 'FDM_Warping_Predictor'.


## 8. Reuse the feature store

In [9]:
# Demonstrate that downstream code can load engineered features without
# recalculating them from the raw CSV.
reused_features = pd.read_csv("feature_store_v1.csv")
X_reused = reused_features[feature_columns]
print("Loaded reusable feature store:", X_reused.shape)
display(X_reused.head())

Loaded reusable feature store: (400, 10)


,nozzle_temperature_c,bed_temperature_c,printing_speed_mm_s,ambient_temperature_c,cooling_fan_speed_pct,component_base_area_cm2,layer_height_mm,nozzle_bed_temperature_difference,cooling_stress_index,filament_material
0,253.4,112.6,51.5,20.8,43.0,54.7,0.30,140.8,338.090133,ABS
1,210.0,59.1,57.7,18.9,51.5,138.4,0.25,150.9,237.415120,PLA
2,201.9,57.3,54.7,18.7,64.6,92.3,0.20,144.6,243.604873,PLA
3,232.3,74.7,51.3,24.6,69.3,80.0,0.10,157.6,358.068146,PETG
4,236.5,102.1,47.0,21.6,40.5,161.8,0.10,134.4,309.999200,ABS


## 9. Test a single future printing job

In [10]:
new_job = pd.DataFrame([{
    "filament_material": "PLA",
    "nozzle_temperature_c": 205,
    "bed_temperature_c": 60,
    "printing_speed_mm_s": 45,
    "ambient_temperature_c": 24,
    "cooling_fan_speed_pct": 60,
    "component_base_area_cm2": 40,
    "layer_height_mm": 0.20
}])
new_job_features = engineer_features(new_job)[feature_columns]
prediction = best_model.predict(new_job_features)[0]
probabilities = best_model.predict_proba(new_job_features)[0]
yes_index = list(best_model.classes_).index("Yes")
risk_probability = float(probabilities[yes_index])
result_text = "Warping Risk" if prediction == "Yes" else "Print Conditions Acceptable"
print({
    "prediction": prediction,
    "result": result_text,
    "warping_probability": round(risk_probability, 4)
})

{'prediction': 'No', 'result': 'Print Conditions Acceptable', 'warping_probability': 0.346}


## 10. FastAPI endpoint demonstration (inside Colab)

In [11]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

app = FastAPI(title="FDM Warping Defect Predictor")

class PrintingJob(BaseModel):
    filament_material: str
    nozzle_temperature_c: float = Field(gt=0, le=350)
    bed_temperature_c: float = Field(ge=0, le=150)
    printing_speed_mm_s: float = Field(gt=0, le=300)
    ambient_temperature_c: float = Field(ge=-10, le=60)
    cooling_fan_speed_pct: float = Field(ge=0, le=100)
    component_base_area_cm2: float = Field(gt=0, le=2000)
    layer_height_mm: float = Field(gt=0, le=1)

@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": best_model is not None}

@app.post("/predict")
def predict(job: PrintingJob):
    if job.filament_material not in {"PLA", "ABS", "PETG"}:
        from fastapi import HTTPException
        raise HTTPException(status_code=422, detail="Material must be PLA, ABS or PETG")
    raw = pd.DataFrame([job.model_dump()])
    features = engineer_features(raw)[feature_columns]
    pred = best_model.predict(features)[0]
    probs = best_model.predict_proba(features)[0]
    p_yes = float(probs[list(best_model.classes_).index("Yes")])
    return {
        "prediction": pred,
        "result": "Warping Risk" if pred == "Yes" else "Print Conditions Acceptable",
        "warping_probability": round(p_yes, 4)
    }

client = TestClient(app)
print("Health:", client.get("/health").json())

sample_payload = new_job.iloc[0].to_dict()
api_response = client.post("/predict", json=sample_payload)
print("Prediction API status:", api_response.status_code)
print("Prediction API response:", api_response.json())
assert api_response.status_code == 200

Health: {'status': 'healthy', 'model_loaded': True}
Prediction API status: 200
Prediction API response: {'prediction': 'No', 'result': 'Print Conditions Acceptable', 'warping_probability': 0.346}


## 11. Download outputs and screenshots

In [12]:
from google.colab import files
for output_file in [
    "feature_store_v1.csv", "model_comparison.csv",
    "artifacts/best_model.joblib", "mlflow.db"
]:
    if Path(output_file).exists():
        files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>